In [2]:
from pathlib import Path
import sys

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = (
        PROJECT_ROOT.parent
    )

sys.path.insert(
    0,
    str(PROJECT_ROOT)
)

In [9]:
from src.transaction_processor import (
    TransactionProcessor
)

In [10]:
from src.replay import TemporalReplay

from src.persistence import (
    buscar_run_replay,
    contar_transacoes,
    buscar_ultimas_transacoes
)

In [3]:
replay = TemporalReplay(
    intervalo_segundos=0.2
)

In [4]:
import pandas as pd


DATA_PATH = (
    PROJECT_ROOT
    / "raw"
    / "fraudTrain.csv"
)


dtypes = {
    "cc_num": "string",
    "trans_num": "string",
    "zip": "string",
    "merchant": "string",
    "category": "category",
    "gender": "category",
    "state": "category",
    "first": "string",
    "last": "string",
    "street": "string",
    "city": "string",
    "job": "string"
}


df = pd.read_csv(
    DATA_PATH,
    dtype=dtypes,
    parse_dates=[
        "trans_date_trans_time",
        "dob"
    ]
)

In [5]:
colunas_indice = [
    coluna
    for coluna in df.columns
    if str(coluna).startswith("Unnamed:")
]

if colunas_indice:
    df = df.drop(
        columns=colunas_indice
    )

In [6]:
df = (
    df
    .sort_values(
        "trans_date_trans_time"
    )
    .reset_index(drop=True)
)

In [7]:
n_total = len(df)

fim_treino = int(
    n_total * 0.70
)

fim_validacao = int(
    n_total * 0.85
)

df_teste = (
    df.iloc[
        fim_validacao:
    ]
    .copy()
)

In [8]:
resultado_replay = (
    replay.executar(
        transacoes=df_teste,
        run_id="replay_teste_20_v1",
        limite=20,
        atualizar_estado_a_cada=1
    )
)

resultado_replay

{'run_id': 'replay_teste_20_v1',
 'status': 'COMPLETED',
 'total_transacoes': 20,
 'processadas': 20,
 'intervalo_segundos': 0.2,
 'tempo_total_segundos': 4.488579208002193,
 'transacoes_por_segundo': 4.455752939447789}

In [9]:
buscar_run_replay(
    "replay_teste_20_v1"
)

{'run_id': 'replay_teste_20_v1',
 'status': 'COMPLETED',
 'total_transacoes': 20,
 'processadas': 20,
 'last_index': 19,
 'intervalo_segundos': 0.2,
 'started_at': '2026-08-17T01:36:56.676544+00:00',
 'updated_at': '2026-08-17T01:37:01.181271+00:00',
 'completed_at': '2026-08-17T01:37:01.181271+00:00',
 'error_message': None,
 'model_version': 'catboost_v1',
 'policy_version': 'decision_policy_v1'}

In [10]:
contar_transacoes(
    run_id="replay_teste_20_v1"
)

20

In [11]:
ultimas = (
    buscar_ultimas_transacoes(
        limite=5,
        run_id="replay_teste_20_v1"
    )
)

pd.DataFrame(
    ultimas
)[
    [
        "trans_date_trans_time",
        "score_fraude",
        "decisao",
        "is_fraud",
        "processed_at"
    ]
]

,trans_date_trans_time,score_fraude,decisao,is_fraud,processed_at
0,2020-04-03T18:08:15,0.000106,APROVAR,0,2026-08-17T01:37:01.173983+00:00
1,2020-04-03T18:06:27,0.000005,APROVAR,0,2026-08-17T01:37:00.947080+00:00
2,2020-04-03T18:05:56,0.000573,APROVAR,0,2026-08-17T01:37:00.701817+00:00
3,2020-04-03T18:04:43,0.000715,APROVAR,0,2026-08-17T01:37:00.459820+00:00
4,2020-04-03T18:01:17,0.000063,APROVAR,0,2026-08-17T01:37:00.224933+00:00


In [2]:
from src.dashboard_queries import (
    carregar_snapshot_dashboard
)

In [3]:
snapshot = (
    carregar_snapshot_dashboard(
        "replay_teste_20_v1"
    )
)

In [4]:
snapshot["replay"]

{'run_id': 'replay_teste_20_v1',
 'status': 'COMPLETED',
 'total_transacoes': 20,
 'processadas': 20,
 'last_index': 19,
 'intervalo_segundos': 0.2,
 'started_at': '2026-08-17T01:36:56.676544+00:00',
 'updated_at': '2026-08-17T01:37:01.181271+00:00',
 'completed_at': '2026-08-17T01:37:01.181271+00:00',
 'model_version': 'catboost_v1',
 'policy_version': 'decision_policy_v1',
 'progresso': 1.0}

In [5]:
snapshot["resumo"]

{'total_processadas': 20, 'aprovadas': 18, 'revisar': 1, 'alertas_criticos': 1}

In [6]:
snapshot["fraudes"]

{'fraudes_reais': 1,
 'fraudes_detectadas': 1,
 'fraudes_perdidas': 0,
 'legitimas_encaminhadas': 1,
 'recall_politica': 1.0}

In [7]:
pd.DataFrame(
    snapshot[
        "transacoes_recentes"
    ]
)

,trans_num,trans_date_trans_time,score_fraude,decisao,is_fraud,latency_ms,processed_at
0,5bfa92b131c53438d92c8bc4317fb102,2020-04-03T18:08:15,0.000106,APROVAR,0,4.947875,2026-08-17T01:37:01.173983+00:00
1,faa838d65abea26aae6913419a3885dc,2020-04-03T18:06:27,0.000005,APROVAR,0,14.734458,2026-08-17T01:37:00.947080+00:00
2,882a4207da708883adb76deb7b4025b9,2020-04-03T18:05:56,0.000573,APROVAR,0,13.929375,2026-08-17T01:37:00.701817+00:00
3,7eb9928dcedc33a087458131ba38c648,2020-04-03T18:04:43,0.000715,APROVAR,0,11.552458,2026-08-17T01:37:00.459820+00:00
4,8893d6d8c5cc747791ddea2ce048932b,2020-04-03T18:01:17,0.000063,APROVAR,0,13.019083,2026-08-17T01:37:00.224933+00:00
5,a317eca6b8fa32c4a557d48543e4d239,2020-04-03T18:01:03,0.000041,APROVAR,0,15.639084,2026-08-17T01:36:59.985680+00:00
6,ce56ee1dc1e69d7772d54eaf0ca89b2b,2020-04-03T18:00:37,0.000041,APROVAR,0,14.371958,2026-08-17T01:36:59.741311+00:00
7,4d66074c315f7eec2c98258fbd67904e,2020-04-03T17:59:58,0.214532,REVISAR,0,11.066583,2026-08-17T01:36:59.500493+00:00
8,ab65d226bbdb804fc2357098ab266267,2020-04-03T17:59:37,0.000075,APROVAR,0,13.082875,2026-08-17T01:36:59.264025+00:00
9,facc072d7797ab112fd9933c822e0184,2020-04-03T17:59:33,0.001064,APROVAR,0,14.970625,2026-08-17T01:36:59.022761+00:00


In [2]:
from src.persistence import (
    inicializar_banco
)

inicializar_banco()

PosixPath('/Users/lucassantos/Documents/ccard_fraud_ml/runtime/fraud_detection.db')

In [3]:
from src.persistence import (
    conectar_banco
)

with conectar_banco() as conexao:

    estrutura = conexao.execute(
        """
        PRAGMA table_info(
            transacoes_processadas
        )
        """
    ).fetchall()

[
    linha["name"]
    for linha in estrutura
]

['id',
 'run_id',
 'trans_num',
 'trans_date_trans_time',
 'score_fraude',
 'decisao',
 'is_fraud',
 'processed_at',
 'latency_ms',
 'model_version',
 'policy_version',
 'first',
 'last',
 'merchant',
 'category',
 'amt',
 'city',
 'state',
 'cc_last4']

In [8]:
transacao_teste = (
    df_teste
    .iloc[[0]]
    .copy()
)

In [11]:
processador = (
    TransactionProcessor()
)

In [12]:
resultado = (
    processador
    .processar_transacao(
        transacao=transacao_teste,
        run_id="teste_contexto_v1"
    )
)

resultado

{'run_id': 'teste_contexto_v1',
 'trans_num': 'b3cb6fd853393499393f26e9285d271f',
 'trans_date_trans_time': Timestamp('2020-04-03 17:54:44'),
 'first': 'Joseph',
 'last': 'Gonzalez',
 'merchant': 'fraud_Schmidt and Sons',
 'category': 'shopping_net',
 'amt': 5.58,
 'city': 'Murfreesboro',
 'state': 'TN',
 'cc_last4': '2969',
 'score_fraude': 2.055211055152726e-05,
 'decisao': 'APROVAR',
 'is_fraud': 0,
 'latency_ms': 20.983875001547858,
 'persistido': True}

In [19]:
from src.dashboard_queries import (
    carregar_snapshot_dashboard
)

snapshot = carregar_snapshot_dashboard(
    "replay_completo_enriquecido_v1"
)

In [20]:
pd.DataFrame(
    snapshot[
        "transacoes_recentes"
    ]
)

,trans_num,trans_date_trans_time,first,last,merchant,category,amt,city,state,cc_last4,score_fraude,decisao,is_fraud,latency_ms,processed_at
0,409ee9f732b35757a740814b35df0661,2020-04-05T14:59:18,Christine,Shaffer,"fraud_O'Reilly, Mohr and Purdy",home,21.97,Loami,IL,6866,0.000231,APROVAR,0,6.753875,2026-08-18T02:18:31.540501+00:00
1,684eb04b8d07cbcef0809eed8de0ee6a,2020-04-05T14:58:48,Kenneth,Sanchez,fraud_Welch Inc,misc_net,153.71,Tekoa,WA,8072,0.000052,APROVAR,0,6.563667,2026-08-18T02:18:31.532041+00:00
2,dd5d81b4b0945b38577c4102739c91a2,2020-04-05T14:58:04,Jerry,Kelly,fraud_Goyette-Herzog,travel,6.56,Fairview,NJ,9857,0.000412,APROVAR,0,4.363375,2026-08-18T02:18:31.524321+00:00
3,a7f3720dd228d296d01a5ae70f757247,2020-04-05T14:58:01,John,Stevens,"fraud_Lakin, Ferry and Beatty",food_dining,3.58,Hudson,NY,6329,0.000004,APROVAR,0,3.485291,2026-08-18T02:18:31.518981+00:00
4,e84ad9af3ea51ca81049448ddbbf1876,2020-04-05T14:57:56,Misty,Rivera,fraud_Spencer PLC,entertainment,26.49,Catawba,VA,0592,0.000024,APROVAR,0,3.434042,2026-08-18T02:18:31.514713+00:00
5,2efa2c486502883423cbc9cf0a0cdc4f,2020-04-05T14:57:54,Bradley,Martinez,"fraud_Schroeder, Hauck and Treutel",entertainment,68.90,Burns Flat,OK,1591,0.000091,APROVAR,0,3.434959,2026-08-18T02:18:31.510527+00:00
6,a059b485053df65906adb930764c2b3f,2020-04-05T14:57:15,Erik,Stevens,"fraud_Conroy, Balistreri and Gorczany",health_fitness,6.38,Lakeland,FL,9415,0.000380,APROVAR,0,3.933833,2026-08-18T02:18:31.506272+00:00
7,38a779963c99e267863c365d61de1006,2020-04-05T14:57:08,John,Chandler,fraud_Collier LLC,home,156.01,Detroit,MI,0508,0.000014,APROVAR,0,3.359208,2026-08-18T02:18:31.501568+00:00
8,564b24087a54dbece3c3f0f21b14cb55,2020-04-05T14:55:45,Kathryn,Smith,"fraud_Streich, Rolfson and Wilderman",kids_pets,75.43,Rocky Mount,MO,5713,0.000057,APROVAR,0,3.377791,2026-08-18T02:18:31.497405+00:00
9,86173f6f4f8c5ea921106d56953982b0,2020-04-05T14:54:51,Andrew,Patterson,fraud_Schuppe-Schuppe,food_dining,50.54,Thida,AR,8427,0.000018,APROVAR,0,3.266250,2026-08-18T02:18:31.493307+00:00


In [16]:
print(len(df_teste))

194502


In [21]:
from src.replay import TemporalReplay

replay_completo = TemporalReplay(
    intervalo_segundos=0
)

RUN_ID = "replay_completo_enriquecido_v2"

resultado_replay_completo = (
    replay_completo.executar(
        transacoes=df_teste,
        run_id=RUN_ID,
        limite=None,
        atualizar_estado_a_cada=100
    )
)

resultado_replay_completo

{'run_id': 'replay_completo_enriquecido_v2',
 'status': 'COMPLETED',
 'total_transacoes': 194502,
 'processadas': 194502,
 'intervalo_segundos': 0.0,
 'tempo_total_segundos': 897.7657427910017,
 'transacoes_por_segundo': 216.6511716022113}

In [23]:
from src.dashboard_queries import (
    buscar_resumo_execucao
)

In [24]:
buscar_resumo_execucao(
    RUN_ID
)

{'total_processadas': 194502,
 'aprovadas': 193425,
 'revisar': 338,
 'alertas_criticos': 739}

In [3]:
from src.dashboard_queries import (
    carregar_snapshot_dashboard
)

RUN_ID = (
    "replay_completo_enriquecido_v2"
)

snapshot = (
    carregar_snapshot_dashboard(
        RUN_ID
    )
)

In [4]:
snapshot.keys()

dict_keys(['replay', 'resumo', 'fraudes', 'valores', 'latencia', 'transacoes_recentes', 'alertas_criticos', 'atividade_risco', 'top_categorias_risco', 'top_categorias_criticas'])